In [1]:
!pip install -q transformers datasets accelerate

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [33]:
# Load your dataset after adding it to your Kaggle notebook
# For this example, I'll assume you saved the combined CSV as 'fake_news.csv'
df = pd.read_csv('/kaggle/input/datasets/khushikyad001/fake-news-detection/fake_news_dataset.csv') 

# If your dataset has separate 'real' and 'fake' CSVs, you can combine them like this:
# df_fake = pd.read_csv('/kaggle/input/your-fake-news-dataset/Fake.csv')
# df_real = pd.read_csv('/kaggle/input/your-fake-news-dataset/True.csv')
# df = pd.concat([df_fake, df_real], ignore_index=True)

# For text generation, we need to work with the 'text' column, so let's see what our columns are
print(df.columns)

# We'll create a new column that combines the title and text for more context
df['full_text'] = "Title: " + df['title'].astype(str) + "\n\nText: " + df['text'].astype(str)

# Let's save this combined text to a simple text file, which is the format our tokenizer expects
with open('/kaggle/working/toxic_train.txt', 'w') as f:
    for text in df['full_text']:
        f.write(text + "\n\n")

Index(['id', 'title', 'author', 'text', 'state', 'date_published', 'source',
       'category', 'sentiment_score', 'word_count', 'char_count', 'has_images',
       'has_videos', 'readability_score', 'num_shares', 'num_comments',
       'political_bias', 'fact_check_rating', 'is_satirical', 'trust_score',
       'source_reputation', 'clickbait_score', 'plagiarism_score', 'label'],
      dtype='object')


In [34]:
df['full_text']

0       Title: Breaking News 1\n\nText: This is the co...
1       Title: Breaking News 2\n\nText: This is the co...
2       Title: Breaking News 3\n\nText: This is the co...
3       Title: Breaking News 4\n\nText: This is the co...
4       Title: Breaking News 5\n\nText: This is the co...
                              ...                        
3995    Title: Breaking News 3996\n\nText: This is the...
3996    Title: Breaking News 3997\n\nText: This is the...
3997    Title: Breaking News 3998\n\nText: This is the...
3998    Title: Breaking News 3999\n\nText: This is the...
3999    Title: Breaking News 4000\n\nText: This is the...
Name: full_text, Length: 4000, dtype: object

In [48]:
df.head(2)

,id,title,author,text,state,date_published,source,category,sentiment_score,word_count,...,num_comments,political_bias,fact_check_rating,is_satirical,trust_score,source_reputation,clickbait_score,plagiarism_score,label,full_text
0,1,Breaking News 1,Jane Smith,This is the content of article 1. It contains ...,Tennessee,30-11-2021,The Onion,Entertainment,-0.22,1302,...,450,Center,FALSE,1,76,6,0.84,53.35,Fake,Title: Breaking News 1\n\nText: This is the co...
1,2,Breaking News 2,Emily Davis,This is the content of article 2. It contains ...,Wisconsin,02-09-2021,The Guardian,Technology,0.92,322,...,530,Left,Mixed,1,1,5,0.85,28.28,Fake,Title: Breaking News 2\n\nText: This is the co...


In [58]:
texts = [
    "The president announced a new tax reform that will benefit middle-class families.",
    "Scientists have discovered a breakthrough in renewable energy storage.",
    "A major cyber attack hit the healthcare system, compromising patient data."
]

with open('/kaggle/working/toxic_train.txt', 'w') as f:
    for text in texts:
        f.write(text.replace('\n', ' ') + "\n")

In [59]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset('text', data_files={'train': '/kaggle/working/toxic_train.txt'})

def tokenize_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [60]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [76]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/gpt2-fake-news",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,            # Log every single step
    save_strategy="epoch",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset['train'],
)

In [77]:
trainer.train()
trainer.save_model("/kaggle/working/gpt2-fake-news-final")
tokenizer.save_pretrained("/kaggle/working/gpt2-fake-news-final")
print("Saved.")

Step,Training Loss
1,0.575711
2,0.592927
3,0.466329


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


In [64]:
# Load our fine-tuned model
model = AutoModelForCausalLM.from_pretrained("/kaggle/working/gpt2-fake-news-final")
tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/gpt2-fake-news-final")

# Set the model to evaluation mode
model.eval()
# Our starting prompt
prompt = "The government recently announced a new policy that"

# Tokenize the input
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text
outputs = model.generate(
    inputs['input_ids'],
    max_new_tokens=150, # The maximum number of tokens to generate
    do_sample=True, # Use sampling to introduce randomness
    temperature=0.8, # Controls the 'creativity' (0 = deterministic, 1 = random)
    top_k=50, # Only sample from the top 50 most likely words
    top_p=0.95, # Only sample from the top 95% of the probability mass
    pad_token_id=tokenizer.eos_token_id 
)

# Decode and print the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

The government recently announced a new policy that will ensure that foreign nationals who travel to the United States cannot be denied visas to stay in the country, including those who have lived in the country since 2012.





The US has announced new policies that will ensure that foreign nationals who travel to the United States cannot be denied visas to stay in the country.
The new policy will ensure that foreign nationals who travel to the United States cannot be denied visas to stay in the country.
In a statement released by the American Immigration Lawyers Association, the government said the new policy will ensure that foreign nationals who travel to the United States cannot be denied visas to stay in the country.
The government said the new policy will ensure that foreign nationals who travel to the United States cannot be


In [75]:
import os
path = "/kaggle/working/gpt2-fake-news-final"
if os.path.exists(path):
    print("Files in directory:", os.listdir(path))
else:
    print("Directory not found")

Files in directory: ['tokenizer.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'config.json']
